# Step 3: Constructed dataset (UCI household power + injected anomalies)
Builds my **new, self-constructed dataset** as described in the proposal (Section 5.2):
a real but unlabelled smart-home sensor stream, with anomalies of known **type, timing and severity**
(spike, drift, dropout, correlated-channel decoupling) injected into the test period. The injection log is the ground truth.

Pipeline: **1. Collection → 2. Preprocessing → 3. Annotation (injection + labels) → 4. Validation figures → 5. (bonus) first USAD run**

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

## 1. Collection

In [ ]:
!wget -q -O uci_power.zip "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
!unzip -o -q uci_power.zip
!ls -la household_power_consumption.txt

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED = 42
rng = np.random.default_rng(SEED)

raw = pd.read_csv('household_power_consumption.txt', sep=';', na_values='?', low_memory=False)
raw['datetime'] = pd.to_datetime(raw['Date'] + ' ' + raw['Time'], format='%d/%m/%Y %H:%M:%S')
raw = raw.drop(columns=['Date', 'Time']).set_index('datetime').astype(float)
CH = list(raw.columns)

print('RAW DATASET')
print('  rows (1-minute readings):', f'{len(raw):,}')
print('  channels:', len(CH), CH)
print('  period:', raw.index.min(), '->', raw.index.max())
print('  missing rows:', f'{raw.isna().any(axis=1).sum():,}', f'({100*raw.isna().any(axis=1).mean():.2f}%)')
print('  labels: none (unlabelled stream)')
raw.describe().T.round(2)

## 2. Preprocessing
- Resample 1-minute readings to **10-minute means** (≈207k steps; keeps daily patterns, makes training fast)
- Fill short gaps (≤ 1 hour) by time interpolation; longer outages are interpolated but **marked as imputed** and excluded from anomaly injection
- Chronological split: **train** = 2007-01-01 to 2009-06-30 (kept clean, normal only), **test** = 2009-07-01 onwards (anomalies injected here)

In [ ]:
FREQ = '10min'
df = raw.resample(FREQ).mean()
missing_before = df.isna().any(axis=1)
df = df.interpolate(method='time', limit=6)                 # gaps up to 1 hour
long_gap = df.isna().any(axis=1)                            # still missing => long outage
df = df.interpolate(method='time').bfill().ffill()
imputed = long_gap.values

train_df = df.loc['2007-01-01':'2009-06-30'].copy()
test_clean = df.loc['2009-07-01':].copy()
test_imputed = imputed[df.index >= pd.Timestamp('2009-07-01')]

print('PREPROCESSED')
print('  steps after 10-min resampling:', f'{len(df):,}')
print('  steps with any missing value before filling:', f'{missing_before.sum():,}', f'({100*missing_before.mean():.2f}%)')
print('  steps inside long outages (>1h, excluded from injection):', f'{long_gap.sum():,}')
print('  train:', train_df.index.min().date(), '->', train_df.index.max().date(), f'({len(train_df):,} steps)')
print('  test :', test_clean.index.min().date(), '->', test_clean.index.max().date(), f'({len(test_clean):,} steps)')
print('\nCorrelation of Global_active_power with Global_intensity (train):',
      round(train_df['Global_active_power'].corr(train_df['Global_intensity']), 3))

## 3. Annotation: injecting anomalies with known type, timing and severity
| Type | What happens | Length (10-min steps) | Severity |
|---|---|---|---|
| **spike** | one channel jumps by k·σ | 1–3 | low 3σ / high 6σ |
| **drift** | one channel slowly offsets, linear ramp up to k·σ | 36–144 (6–24 h) | low 2σ / high 4σ |
| **dropout** | channel(s) read 0 (meter outage) | 12–72 (2–12 h) | low = 1 channel / high = all channels |
| **decoupling** | Global_intensity replaced by its value from another day, breaking its link to Global_active_power | 24–72 (4–12 h) | low = 50% mix / high = 100% |

σ = channel standard deviation on the training period. Segments never overlap, keep ≥ 1 day apart, and avoid imputed outages.

In [ ]:
N_PER_TYPE = 20
LENGTHS = {'spike': (1, 3), 'drift': (36, 144), 'dropout': (12, 72), 'decoupling': (24, 72)}
GAP = 144                                     # >= 1 day between segments
sigma = train_df.std()

X = test_clean.values.copy()
n = len(X)
occupied = test_imputed.copy()                # do not inject into imputed outages
labels = np.zeros(n, dtype=int)
log = []
ci = {c: i for i, c in enumerate(CH)}

def find_slot(length):
    for _ in range(10000):
        s = int(rng.integers(GAP, n - length - GAP))
        if not occupied[s - GAP:s + length + GAP].any():
            occupied[s - GAP:s + length + GAP] = True
            return s
    raise RuntimeError('no free slot')

types = [t for t in LENGTHS for _ in range(N_PER_TYPE)]
rng.shuffle(types)
for k, t in enumerate(types):
    L = int(rng.integers(LENGTHS[t][0], LENGTHS[t][1] + 1))
    s = find_slot(L); e = s + L
    level = rng.choice(['low', 'high'])
    if t == 'spike':
        ch = rng.choice(CH); mag = (3 if level == 'low' else 6) * sigma[ch]
        sign = 1 if ch != 'Voltage' else rng.choice([-1, 1])
        X[s:e, ci[ch]] += sign * mag; chans = ch
    elif t == 'drift':
        ch = rng.choice(CH); mag = (2 if level == 'low' else 4) * sigma[ch]
        sign = 1 if ch != 'Voltage' else rng.choice([-1, 1])
        X[s:e, ci[ch]] += sign * np.linspace(0, mag, L); chans = ch
    elif t == 'dropout':
        if level == 'low':
            ch = rng.choice(CH); X[s:e, ci[ch]] = 0; chans = ch
        else:
            X[s:e, :] = 0; chans = 'all'
    else:  # decoupling
        j = ci['Global_intensity']
        src = int(rng.integers(0, n - L))
        while abs(src - s) < 1008:            # take the replacement from >= 1 week away
            src = int(rng.integers(0, n - L))
        w = 0.5 if level == 'low' else 1.0
        X[s:e, j] = (1 - w) * X[s:e, j] + w * test_clean.values[src:src + L, j]
        chans = 'Global_intensity'
    X = np.clip(X, 0, None)
    labels[s:e] = 1
    log.append({'id': k, 'type': t, 'channel': chans, 'severity': level,
                'start': test_clean.index[s], 'end': test_clean.index[e - 1],
                'start_idx': s, 'end_idx': e, 'length_steps': L})

test_df = pd.DataFrame(X, index=test_clean.index, columns=CH)
test_df['label'] = labels
inj = pd.DataFrame(log).sort_values('start_idx').reset_index(drop=True)

print('ANNOTATED TEST SET')
print('  injected segments:', len(inj))
print('  anomalous steps:', labels.sum(), f'({100*labels.mean():.2f}% of test)')
summary = inj.groupby('type').agg(segments=('id', 'count'), high_severity=('severity', lambda s: (s == 'high').sum()),
                                  mean_len_steps=('length_steps', 'mean'), total_steps=('length_steps', 'sum')).round(1)
summary

## 4. Save the dataset + validation figures

In [ ]:
train_df.to_csv('uci_constructed_train.csv')
test_df.to_csv('uci_constructed_test.csv')
inj.to_csv('injection_log.csv', index=False)
summary.to_csv('injection_summary.csv')

stats = pd.DataFrame([
    {'property': 'Source', 'value': 'UCI Individual Household Electric Power Consumption (Hebrail & Berard, 2012)'},
    {'property': 'Raw size', 'value': f'{len(raw):,} one-minute readings, {len(CH)} channels'},
    {'property': 'After preprocessing', 'value': f'{len(df):,} ten-minute steps'},
    {'property': 'Train (normal only)', 'value': f'{len(train_df):,} steps'},
    {'property': 'Test (injected)', 'value': f'{len(test_df):,} steps'},
    {'property': 'Injected segments', 'value': f'{len(inj)} ({N_PER_TYPE} per type x 4 types)'},
    {'property': 'Anomaly ratio (test)', 'value': f'{100*labels.mean():.2f}%'},
    {'property': 'Random seed', 'value': str(SEED)},
])
stats.to_csv('dataset_card.csv', index=False)
stats

In [ ]:
# Figure: one example of each anomaly type, original (grey) vs injected (blue), segment shaded red
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, t in zip(axes.ravel(), ['spike', 'drift', 'dropout', 'decoupling']):
    r = inj[(inj.type == t) & (inj.severity == 'high')].iloc[0]
    s, e = r.start_idx, r.end_idx
    a, b = max(0, s - 150), min(n, e + 150)
    ch = 'Global_intensity' if r.channel == 'all' else r.channel
    x = test_clean.index[a:b]
    ax.plot(x, test_clean[ch].values[a:b], color='grey', lw=1, label='original')
    ax.plot(x, test_df[ch].values[a:b], color='tab:blue', lw=1, label='with injected anomaly')
    if t == 'decoupling':
        ax.plot(x, test_df['Global_active_power'].values[a:b] * 4.2, color='tab:green', lw=.8, alpha=.7,
                label='Global_active_power (scaled)')
    ax.axvspan(test_clean.index[s], test_clean.index[e - 1], color='red', alpha=.2, label='label = 1')
    ax.set_title(f'{t} ({r.severity}) on {r.channel}'); ax.tick_params(axis='x', labelrotation=20, labelsize=8)
    ax.legend(fontsize=7, loc='upper right')
plt.suptitle('Constructed dataset: injected anomaly types', fontsize=13)
plt.tight_layout(); plt.savefig('fig_injection_examples.png', dpi=150); plt.show()

In [ ]:
# Figure: where the anomalies are across the whole test period
colors = {'spike': 'tab:red', 'drift': 'tab:orange', 'dropout': 'tab:purple', 'decoupling': 'tab:green'}
plt.figure(figsize=(13, 3.5))
plt.plot(test_df.index, test_df['Global_active_power'], lw=.3, color='grey')
for t, col in colors.items():
    for _, r in inj[inj.type == t].iterrows():
        plt.axvspan(r.start, r.end + pd.Timedelta(FREQ), color=col, alpha=.5)
    plt.plot([], [], color=col, lw=6, label=t)
plt.ylabel('Global_active_power (kW)'); plt.title('Test period with 80 injected anomaly segments')
plt.legend(ncol=4, loc='upper right', fontsize=8); plt.tight_layout(); plt.savefig('fig_injection_timeline.png', dpi=150); plt.show()

## 5. (Bonus) First USAD run on the constructed dataset, with detection rate by anomaly type

In [ ]:
!git clone -q https://github.com/manigalati/usad.git
import sys, time, torch, torch.utils.data as data_utils
sys.path.append('/content/usad')
from usad import *
from utils import get_default_device, to_device
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
device = get_default_device(); torch.manual_seed(0)
WINDOW_SIZE, HIDDEN_SIZE, BATCH_SIZE, N_EPOCHS = 12, 100, 7919, 100

def make_windows(x, w):
    idx = np.arange(w)[None, :] + np.arange(x.shape[0] - w)[:, None]
    return x[idx]

def point_adjust(pred, label):
    pred = pred.copy(); i, m = 0, len(label)
    while i < m:
        if label[i] == 1:
            j = i
            while j < m and label[j] == 1: j += 1
            if pred[i:j].any(): pred[i:j] = 1
            i = j
        else: i += 1
    return pred

def best_f1(score, label, adjust):
    best = (0, 0, 0, 0)
    for th in np.quantile(score, np.linspace(0.5, 0.9999, 300)):
        pred = (score >= th).astype(int)
        if adjust: pred = point_adjust(pred, label)
        p, r, f, _ = precision_recall_fscore_support(label, pred, average='binary', zero_division=0)
        if f > best[2]: best = (p, r, f, th)
    return best

scaler = MinMaxScaler().fit(train_df[CH].values)
tr = scaler.transform(train_df[CH].values)
te = np.clip(scaler.transform(test_df[CH].values), -5, 5)
w_tr, w_te = make_windows(tr, WINDOW_SIZE), make_windows(te, WINDOW_SIZE)
win_lab = np.array([1 if labels[i:i+WINDOW_SIZE].sum() > 0 else 0 for i in range(len(w_te))])
w_size, z_size = WINDOW_SIZE * len(CH), WINDOW_SIZE * HIDDEN_SIZE
to_t = lambda a: torch.from_numpy(a).float().view(-1, w_size)
split = int(0.8 * len(w_tr))
tl = data_utils.DataLoader(data_utils.TensorDataset(to_t(w_tr[:split])), batch_size=BATCH_SIZE, shuffle=False)
vl = data_utils.DataLoader(data_utils.TensorDataset(to_t(w_tr[split:])), batch_size=BATCH_SIZE, shuffle=False)
xl = data_utils.DataLoader(data_utils.TensorDataset(to_t(w_te)), batch_size=BATCH_SIZE, shuffle=False)

model = to_device(UsadModel(w_size, z_size), device)
t0 = time.time(); history = training(N_EPOCHS, model, tl, vl)
print(f'Training time: {(time.time()-t0)/60:.1f} min')
with torch.no_grad():
    score = torch.cat([r.flatten() for r in testing(model, xl)]).cpu().numpy()

In [ ]:
raw_r, pa_r = best_f1(score, win_lab, False), best_f1(score, win_lab, True)
res = pd.DataFrame([
    {'setting': 'with point adjustment', 'Precision': pa_r[0], 'Recall': pa_r[1], 'F1': pa_r[2]},
    {'setting': 'without point adjustment', 'Precision': raw_r[0], 'Recall': raw_r[1], 'F1': raw_r[2]}])
print('AUC:', round(roc_auc_score(win_lab, score), 3))
res.to_csv('uci_usad_results.csv', index=False); display(res.round(3))

# Detection rate per anomaly type: a segment counts as detected if any overlapping window is flagged
th = raw_r[3]
flag = score >= th
def detected(r):
    a, b = max(0, r.start_idx - WINDOW_SIZE + 1), min(len(flag), r.end_idx)
    return bool(flag[a:b].any())
inj['detected'] = inj.apply(detected, axis=1)
by_type = inj.groupby(['type', 'severity']).detected.mean().unstack().round(2)
by_type['overall'] = inj.groupby('type').detected.mean().round(2)
by_type.to_csv('uci_detection_by_type.csv'); by_type

In [ ]:
from google.colab import files
for f in ['dataset_card.csv', 'injection_summary.csv', 'injection_log.csv', 'fig_injection_examples.png',
          'fig_injection_timeline.png', 'uci_usad_results.csv', 'uci_detection_by_type.csv']:
    files.download(f)